In [5]:
import re

def extract_kernel_line(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Kernel(" in line:
                return line.strip()
    raise ValueError("No kernel specification found in the file.")

def extract_noise_parameter(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Kernel(" in line or "ScoredKernel(" in line:
                match = re.search(r'noise=\[([-0-9eE\.\+]+)\]', line)
                if match:
                    return float(match.group(1))
    raise ValueError("No noise parameter found in the file.")

In [6]:
def parse_kernel_expr(expr):
    expr = expr.strip()
    # SumKernel([ ... ])
    if expr.startswith('SumKernel(['):
        inner = expr[len('SumKernel(['):-1]
        children = split_kernels(inner)
        return {'type': 'sum', 'children': [parse_kernel_expr(child) for child in children]}
    # ProductKernel([ ... ])
    elif expr.startswith('ProductKernel(['):
        inner = expr[len('ProductKernel(['):-1]
        children = split_kernels(inner)
        return {'type': 'product', 'children': [parse_kernel_expr(child) for child in children]}
    # MaskKernel(ndim=..., active_dimension=..., base_kernel=...)
    elif expr.startswith('MaskKernel('):
        ndim = int(re.search(r'ndim=(\d+)', expr).group(1))
        active_dimension = int(re.search(r'active_dimension=(\d+)', expr).group(1))
        base_kernel_match = re.search(r'base_kernel=(.+\))$', expr)
        base_kernel_str = base_kernel_match.group(1)
        return {
            'type': 'mask',
            'ndim': ndim,
            'active_dimension': active_dimension,
            'base_kernel': parse_kernel_expr(base_kernel_str)
        }
    # SqExpKernel(lengthscale=..., output_variance=...)
    elif expr.startswith('SqExpKernel('):
        params = dict(re.findall(r'(\w+)=([-\d\.]+)', expr))
        return {
            'type': 'sqexp', 
            'lengthscale': float(params['lengthscale']),
            'variance': float(params['output_variance'])
        }
    # SqExpPeriodicKernel(lengthscale=..., period=..., output_variance=...)
    elif expr.startswith('SqExpPeriodicKernel('):
        params = dict(re.findall(r'(\w+)=([-\d\.]+)', expr))
        return {
            'type': 'sqexpperiodic', 
            'lengthscale': float(params['lengthscale']),
            'period': float(params['period']),
            'variance': float(params['output_variance'])
        }
    else:
        raise ValueError(f"Unknown kernel: {expr}")

def split_kernels(s):
    children = []
    depth = 0
    last = 0
    for i, c in enumerate(s):
        if c in '([':
            depth += 1
        elif c in ')]':
            depth -= 1
        elif c == ',' and depth == 0:
            children.append(s[last:i].strip())
            last = i+1
    children.append(s[last:].strip())
    return [child for child in children if child]

In [13]:
file_path = '/home1/09909/smata/dir_scratch/induction_modeling/gaussian_process/10MW/results/rotor/wrf_10MW_rot_result.txt'  # Update this!

# 1. Extract kernel line
kernel_line = extract_kernel_line(file_path)
print("Kernel line:")
print(kernel_line)

Kernel line:
ScoredKernel(k_opt=SumKernel([ MaskKernel(ndim=3, active_dimension=1, base_kernel=SqExpKernel(lengthscale=-5.244907, output_variance=-7.637613)), ProductKernel([ MaskKernel(ndim=3, active_dimension=0, base_kernel=SqExpPeriodicKernel(lengthscale=-0.824300, period=-0.148839, output_variance=0.761611)), MaskKernel(ndim=3, active_dimension=1, base_kernel=SqExpPeriodicKernel(lengthscale=1.519882, period=-2.233469, output_variance=-0.147605)), MaskKernel(ndim=3, active_dimension=2, base_kernel=SqExpKernel(lengthscale=-0.559903, output_variance=-4.151602)) ]) ]), nll=-491.644610, laplace_nle=nan, bic_nle=-950.144141, noise=[-11.28078267])


In [15]:
# 1. Extract noise
noise_log = extract_noise_parameter(file_path)
print("Noise:")
print(noise_log)

Noise:
-11.28078267


In [8]:
# 2. Extract the kernel string argument (remove wrapper like k_opt=..., nll=)
kernel_str_match = re.search(r'k_opt=(.+\)), nll=', kernel_line)
kernel_str = kernel_str_match.group(1)
print("\nKernel specification string:")
print(kernel_str)


Kernel specification string:
SumKernel([ MaskKernel(ndim=3, active_dimension=1, base_kernel=SqExpKernel(lengthscale=-5.244907, output_variance=-7.637613)), ProductKernel([ MaskKernel(ndim=3, active_dimension=0, base_kernel=SqExpPeriodicKernel(lengthscale=-0.824300, period=-0.148839, output_variance=0.761611)), MaskKernel(ndim=3, active_dimension=1, base_kernel=SqExpPeriodicKernel(lengthscale=1.519882, period=-2.233469, output_variance=-0.147605)), MaskKernel(ndim=3, active_dimension=2, base_kernel=SqExpKernel(lengthscale=-0.559903, output_variance=-4.151602)) ]) ])


In [9]:
# 3. Parse recursively
parsed_kernel = parse_kernel_expr(kernel_str)
import pprint
pprint.pprint(parsed_kernel)

{'children': [{'active_dimension': 1,
               'base_kernel': {'lengthscale': -5.244907,
                               'type': 'sqexp',
                               'variance': -7.637613},
               'ndim': 3,
               'type': 'mask'},
              {'children': [{'active_dimension': 0,
                             'base_kernel': {'lengthscale': -0.8243,
                                             'period': -0.148839,
                                             'type': 'sqexpperiodic',
                                             'variance': 0.761611},
                             'ndim': 3,
                             'type': 'mask'},
                            {'active_dimension': 1,
                             'base_kernel': {'lengthscale': 1.519882,
                                             'period': -2.233469,
                                             'type': 'sqexpperiodic',
                                             'variance': -0.147605},
      

In [10]:
import numpy as np
from sklearn.gaussian_process.kernels import Kernel
from sklearn.gaussian_process.kernels import WhiteKernel

class MaskedKernel(Kernel):
    """A kernel wrapper to mask input features (dimensions) for any scikit-learn kernel."""
    def __init__(self, base_kernel, active_dims):
        self.base_kernel = base_kernel
        self.active_dims = np.array(active_dims)

    def __call__(self, X, Y=None, eval_gradient=False):
        X_sub = X[:, self.active_dims]
        if Y is not None:
            Y_sub = Y[:, self.active_dims]
        else:
            Y_sub = None
        return self.base_kernel(X_sub, Y_sub, eval_gradient=eval_gradient)

    def diag(self, X):
        X_sub = X[:, self.active_dims]
        return self.base_kernel.diag(X_sub)

    def is_stationary(self):
        return self.base_kernel.is_stationary()

    def __repr__(self):
        return f"MaskedKernel({repr(self.base_kernel)}, active_dims={self.active_dims.tolist()})"

In [11]:
import numpy as np
from sklearn.gaussian_process.kernels import RBF, ExpSineSquared, ConstantKernel as C

def build_sklearn_kernel(parsed, noise_log=None):
    if parsed['type'] == 'sum':
        result = None
        for child in parsed['children']:
            term = build_sklearn_kernel(child)
            result = term if result is None else result + term
    elif parsed['type'] == 'product':
        result = None
        for child in parsed['children']:
            term = build_sklearn_kernel(child)
            result = term if result is None else result * term
    elif parsed['type'] == 'mask':
        active_dim = [parsed['active_dimension']]
        base = build_sklearn_kernel(parsed['base_kernel'])
        result = MaskedKernel(base, active_dim)
    elif parsed['type'] == 'sqexp':
        lengthscale = np.exp(parsed['lengthscale'])
        variance = np.exp(parsed['variance'])
        result = C(variance) * RBF(length_scale=lengthscale)
    elif parsed['type'] == 'sqexpperiodic':
        lengthscale = np.exp(parsed['lengthscale'])
        period = np.exp(parsed['period'])
        variance = np.exp(parsed['variance'])
        result = C(variance) * RBF(length_scale=lengthscale) * ExpSineSquared(length_scale=lengthscale, periodicity=period)
    else:
        raise ValueError(f"Unknown kernel type: {parsed['type']}")

    # If noise_log is provided, add WhiteKernel
    if noise_log is not None:
        noise_level = np.exp(noise_log)
        result = result + WhiteKernel(noise_level=noise_level)
    return result

def set_active_dims(kernel, active_dims):
    """Helper to patch active_dims attribute on kernels for scikit-learn ≤1.2 compatibility."""
    try:
        return kernel.clone_with_theta(kernel.theta, active_dims=active_dims)
    except Exception:
        kernel.active_dims = active_dims  # For kernels where .active_dims exists
        return kernel


In [16]:
my_kernel = build_sklearn_kernel(parsed_kernel, noise_log=noise_log)
print(my_kernel)

MaskedKernel(0.022**2 * RBF(length_scale=0.00527), active_dims=[1]) + MaskedKernel(1.46**2 * RBF(length_scale=0.439) * ExpSineSquared(length_scale=0.439, periodicity=0.862), active_dims=[0]) * MaskedKernel(0.929**2 * RBF(length_scale=4.57) * ExpSineSquared(length_scale=4.57, periodicity=0.107), active_dims=[1]) * MaskedKernel(0.125**2 * RBF(length_scale=0.571), active_dims=[2]) + WhiteKernel(noise_level=1.26e-05)


In [1]:
from wrf_io import postproc

In [2]:
file_path = '/home1/09909/smata/dir_scratch/induction_modeling/gaussian_process/10MW/results/rotor/wrf_10MW_rot_result.txt'  # Update this!

In [3]:
postproc.build_kernel_from_search(file_path=file_path)

Kernel line:
ScoredKernel(k_opt=SumKernel([ MaskKernel(ndim=3, active_dimension=1, base_kernel=SqExpKernel(lengthscale=-5.244907, output_variance=-7.637613)), ProductKernel([ MaskKernel(ndim=3, active_dimension=0, base_kernel=SqExpPeriodicKernel(lengthscale=-0.824300, period=-0.148839, output_variance=0.761611)), MaskKernel(ndim=3, active_dimension=1, base_kernel=SqExpPeriodicKernel(lengthscale=1.519882, period=-2.233469, output_variance=-0.147605)), MaskKernel(ndim=3, active_dimension=2, base_kernel=SqExpKernel(lengthscale=-0.559903, output_variance=-4.151602)) ]) ]), nll=-491.644610, laplace_nle=nan, bic_nle=-950.144141, noise=[-11.28078267])

Noise:
-11.28078267

Kernel specification string:
SumKernel([ MaskKernel(ndim=3, active_dimension=1, base_kernel=SqExpKernel(lengthscale=-5.244907, output_variance=-7.637613)), ProductKernel([ MaskKernel(ndim=3, active_dimension=0, base_kernel=SqExpPeriodicKernel(lengthscale=-0.824300, period=-0.148839, output_variance=0.761611)), MaskKernel(nd

MaskedKernel(0.022**2 * RBF(length_scale=0.00527), active_dims=[1]) + MaskedKernel(1.46**2 * RBF(length_scale=0.439) * ExpSineSquared(length_scale=0.439, periodicity=0.862), active_dims=[0]) * MaskedKernel(0.929**2 * RBF(length_scale=4.57) * ExpSineSquared(length_scale=4.57, periodicity=0.107), active_dims=[1]) * MaskedKernel(0.125**2 * RBF(length_scale=0.571), active_dims=[2]) + WhiteKernel(noise_level=1.26e-05)